# 01 — Data Exploration

Initial exploratory analysis of the Aave V2 liquidation dataset.

**Goal:** understand the data we have to work with before building features.

**Questions to answer:**
1. How many liquidations do we have in total?
2. How many unique borrowers (= our positive class)?
3. What is the temporal distribution of liquidations?
4. Which collateral/debt asset pairs are most common?
5. Are some borrowers liquidated multiple times?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [ ]:
df = pd.read_parquet('../data/raw/aave_v2_liquidations.parquet')
print(f'Total liquidations: {len(df):,}')
print(f'Unique borrowers:  {df["borrower"].nunique():,}')
print(f'Block range:       {df["block_number"].min():,} → {df["block_number"].max():,}')
df.head()

## 1. Repeat liquidations

Some borrowers are liquidated more than once. This is meaningful for the label —
it suggests sustained risky behavior rather than a one-off market crash event.

In [ ]:
liq_counts = df['borrower'].value_counts()
print(f'Borrowers liquidated only once: {(liq_counts == 1).sum():,}')
print(f'Borrowers liquidated 2+ times:  {(liq_counts >= 2).sum():,}')
print(f'Max liquidations on a single wallet: {liq_counts.max()}')

fig, ax = plt.subplots(figsize=(9, 4))
liq_counts.value_counts().sort_index().head(15).plot(kind='bar', ax=ax, color='#185FA5')
ax.set_title('Distribution of liquidations per borrower')
ax.set_xlabel('Number of liquidations')
ax.set_ylabel('Number of wallets')
plt.tight_layout()
plt.show()

## 2. Temporal distribution

Liquidations are highly concentrated around market stress events (e.g. May 2021 crash,
May–June 2022 LUNA collapse, November 2022 FTX). This has implications for the
train/test split strategy — we may want to use temporal splitting rather than random.

In [ ]:
# Bucket by block range (~~ time)
df['block_bucket'] = (df['block_number'] // 100_000) * 100_000

bucket_counts = df.groupby('block_bucket').size()
fig, ax = plt.subplots(figsize=(11, 4))
bucket_counts.plot(ax=ax, color='#185FA5')
ax.set_title('Liquidations over time (by 100k-block buckets)')
ax.set_xlabel('Block number')
ax.set_ylabel('Number of liquidations')
plt.tight_layout()
plt.show()

## 3. Most common asset pairs

Understanding which collateral/debt combinations dominate the dataset helps with
feature engineering — we may want explicit features for popular pairs vs. rare ones.

In [ ]:
pair_counts = (
    df.groupby(['collateral_asset', 'debt_asset'])
      .size()
      .sort_values(ascending=False)
      .head(15)
)
pair_counts.to_frame('count')

## Next steps

- Sample a non-default cohort: wallets that interacted with Aave but were never liquidated
- Pull full transaction history for both cohorts
- Move to `02_feature_engineering.ipynb`